# ✅ CNN vs Vision Transformer on CIFAR-10 (Colab Ready - Fixed)

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers


In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
y_train = y_train.flatten()
y_test = y_test.flatten()


In [ ]:
plt.figure(figsize=(6,6))
for i in range(9):
    plt.subplot(3,3,i+1)
    plt.imshow(x_train[i])
    plt.title(y_train[i])
    plt.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
unique, counts = np.unique(y_train, return_counts=True)
plt.bar(unique, counts)
plt.title('Class Distribution')
plt.show()


## ✅ CNN Model

In [ ]:
cnn = keras.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)),
    layers.MaxPooling2D(),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])
cnn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cnn.summary()


In [ ]:
history_cnn = cnn.fit(x_train, y_train, epochs=5, batch_size=32, validation_split=0.2)


## ✅ Vision Transformer (Simplified & Fixed)

In [ ]:
patch_size = 4
num_patches = (32 // patch_size) ** 2
projection_dim = 64


In [ ]:
inputs = layers.Input(shape=(32,32,3))

# Extract patches properly
patches = tf.image.extract_patches(
    images=inputs,
    sizes=[1, patch_size, patch_size, 1],
    strides=[1, patch_size, patch_size, 1],
    rates=[1, 1, 1, 1],
    padding='VALID'
)

patches = layers.Reshape((num_patches, patch_size*patch_size*3))(patches)

x = layers.Dense(projection_dim)(patches)
x = layers.MultiHeadAttention(num_heads=4, key_dim=projection_dim)(x, x)
x = layers.Flatten()(x)
x = layers.Dense(64, activation='relu')(x)
outputs = layers.Dense(10, activation='softmax')(x)

vit = keras.Model(inputs, outputs)
vit.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
vit.summary()


In [ ]:
history_vit = vit.fit(x_train, y_train, epochs=5, batch_size=32, validation_split=0.2)
